In [2]:
import os
import numpy as np
import pandas as pd
import joblib

# Regression Models
from sklearn.linear_model import Ridge
# Evaluation Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import optuna

In [3]:
# Setup path
BASE_PATH = 'dataset'
OUTPUT_FOLDER = os.path.join(BASE_PATH, 'output_pipeline')
ARTIFACTS_FOLDER = os.path.join(BASE_PATH, 'artifacts')
MODEL_FOLDER = os.path.join(BASE_PATH, 'model_outputs')
EDA_FOLDER = os.path.join(BASE_PATH, 'eda_outputs')
RESULTS_FOLDER = os.path.join(BASE_PATH, 'results')

os.makedirs(MODEL_FOLDER, exist_ok=True)
os.makedirs(EDA_FOLDER, exist_ok=True)
os.makedirs(RESULTS_FOLDER, exist_ok=True)

In [4]:
# Load fitur (X)
X_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_train.parquet'))
X_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_val.parquet'))
X_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_test.parquet'))

In [5]:
# Load target (y) - convert ke Series untuk sklearn
y_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_train.parquet'))
y_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_val.parquet'))
y_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_test.parquet'))

In [6]:
# Load metadata ID (untuk traceback hasil prediksi nanti)
ID_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_train.parquet'))
ID_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_val.parquet'))
ID_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_test.parquet'))

In [7]:
scaler = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'minmax_scaler.joblib'))
winsor = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'winsor_bounds.joblib'))
feat_select = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'feature_selection_info.joblib'))

/home/aliarridha/miniconda3/envs/modeling1/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [8]:
TARGET = feat_select['target']

In [9]:
y_train_arr = y_train[TARGET].values
y_val_arr = y_val[TARGET].values
y_test_arr = y_test[TARGET].values

In [10]:
print(ID_test.head())
print(ID_test.columns)

         kabupaten_kota   nama_wilayah_bersih  tahun
0       Kab. Aceh Barat       kab. aceh barat   2024
1       Kab. Aceh Barat       kab. aceh barat   2025
2  Kab. Aceh Barat Daya  kab. aceh barat daya   2024
3  Kab. Aceh Barat Daya  kab. aceh barat daya   2025
4       Kab. Aceh Besar       kab. aceh besar   2024
Index(['kabupaten_kota', 'nama_wilayah_bersih', 'tahun'], dtype='object')


In [21]:
print("Mulai tuning Ridge Regression (Bayesian - Optuna)")

def objective(trial):

    params = {
        "alpha": trial.suggest_float(
            "alpha",
            1e-4,
            50,
            log=True
        ),
        "solver": trial.suggest_categorical(
            "solver",
            ["auto", "lsqr", "sag", "cholesky"]
        ),
        "tol": trial.suggest_float(
            "tol",
            1e-6,
            1e-3,
            log=True
        ),
        "fit_intercept": True
    }

    model = Ridge(**params)
    model.fit(X_train, y_train_arr)
    val_pred = model.predict(X_val)
    val_mae = mean_absolute_error(y_val_arr, val_pred)
    return val_mae

study = optuna.create_study(direction="minimize")
study.optimize(
    objective,
    n_trials=150,
    show_progress_bar=True
)

best_params = study.best_params
best_mae = study.best_value

ridge_model = Ridge(**best_params)
ridge_model.fit(X_train, y_train_arr)

print("Tuning selesai")
print("Best params:")
print(best_params)
print(f"Best Validation MAE: {best_mae:.4f}")

[I 2026-06-05 18:07:14,875] A new study created in memory with name: no-name-68c3dc81-43b2-4d1b-bb45-40f815e884af


Mulai tuning Ridge Regression (Bayesian - Optuna)


Best trial: 1. Best value: 4.48632:   1%|          | 1/150 [00:00<00:21,  7.06it/s]

[I 2026-06-05 18:07:15,017] Trial 0 finished with value: 4.509832204082139 and parameters: {'alpha': 0.0002757292824452905, 'solver': 'lsqr', 'tol': 0.0007862294541142064}. Best is trial 0 with value: 4.509832204082139.
[I 2026-06-05 18:07:15,039] Trial 1 finished with value: 4.486324159533616 and parameters: {'alpha': 0.2451866381326673, 'solver': 'auto', 'tol': 7.036595869976172e-06}. Best is trial 1 with value: 4.486324159533616.


Best trial: 3. Best value: 4.47987:   3%|▎         | 4/150 [00:00<00:12, 11.98it/s]

...

[I 2026-06-05 18:07:21,344] Trial 137 finished with value: 4.42177412670603 and parameters: {'alpha': 0.7472177008062615, 'solver': 'lsqr', 'tol': 0.0006255615865355669}. Best is trial 38 with value: 4.375688066227486.
[I 2026-06-05 18:07:21,398] Trial 138 finished with value: 4.509836266051624 and parameters: {'alpha': 0.0002590982877384046, 'solver': 'lsqr', 'tol': 0.0008584411662003686}. Best is trial 38 with value: 4.375688066227486.
[I 2026-06-05 18:07:21,422] Trial 139 finished with value: 4.377375773717832 and parameters: {'alpha': 2.0245897511770274, 'solver': 'lsqr', 'tol': 0.0007015761499601825}. Best is trial 38 with value: 4.375688066227486.
[I 2026-06-05 18:07:21,452] Trial 140 finished with value: 4.404701547737764 and parameters: {'alpha': 1.1543900151872113, 'solver': 'lsqr', 'tol': 0.0006739192114654998}. Best is trial 38 with value: 4.375688066227486.
[I 2026-06-05 18:07:21,481] Trial 141 finished with value: 4.3773807078334155 and parameters: {'alpha': 2.031610150490

Best trial: 38. Best value: 4.37569:  98%|█████████▊| 147/150 [00:06<00:00, 22.51it/s]

[I 2026-06-05 18:07:21,639] Trial 143 finished with value: 4.38219666133847 and parameters: {'alpha': 3.549245489563527, 'solver': 'lsqr', 'tol': 0.0007464097378740812}. Best is trial 38 with value: 4.375688066227486.
[I 2026-06-05 18:07:21,728] Trial 144 finished with value: 4.3861927525403095 and parameters: {'alpha': 2.13265602627772, 'solver': 'lsqr', 'tol': 0.0005935211760537002}. Best is trial 38 with value: 4.375688066227486.
[I 2026-06-05 18:07:21,759] Trial 145 finished with value: 4.377476601495624 and parameters: {'alpha': 2.1790027579561624, 'solver': 'lsqr', 'tol': 0.0007571521854525575}. Best is trial 38 with value: 4.375688066227486.
[I 2026-06-05 18:07:21,777] Trial 146 finished with value: 4.394697233866003 and parameters: {'alpha': 4.213656463300435, 'solver': 'cholesky', 'tol': 0.0005564273616306965}. Best is trial 38 with value: 4.375688066227486.
[I 2026-06-05 18:07:21,796] Trial 147 finished with value: 4.413023428192687 and parameters: {'alpha': 0.842525603358778

Best trial: 38. Best value: 4.37569: 100%|██████████| 150/150 [00:07<00:00, 21.05it/s]

[I 2026-06-05 18:07:21,906] Trial 148 finished with value: 4.3784121637476625 and parameters: {'alpha': 1.263752621046558, 'solver': 'lsqr', 'tol': 0.000849168403007852}. Best is trial 38 with value: 4.375688066227486.
[I 2026-06-05 18:07:21,997] Trial 149 finished with value: 4.380425501726843 and parameters: {'alpha': 3.3031550552162408, 'solver': 'lsqr', 'tol': 0.0007451549641333578}. Best is trial 38 with value: 4.375688066227486.
Tuning selesai
Best params:
{'alpha': 2.4969467753603514, 'solver': 'lsqr', 'tol': 0.0009974758605840026}
Best Validation MAE: 4.3757


In [22]:
# Prediksi
ridge_train_pred = ridge_model.predict(X_train)
ridge_val_pred = ridge_model.predict(X_val)
ridge_test_pred = ridge_model.predict(X_test)

In [23]:
def hitung_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

In [24]:
# evaluasi train
ridge_train_mae, ridge_train_rmse, ridge_train_r2 = hitung_metrics(
    y_train,
    ridge_train_pred
)


In [25]:
# evaluasi validation
ridge_val_mae, ridge_val_rmse, ridge_val_r2 = hitung_metrics(
    y_val,
    ridge_val_pred
)

In [26]:
# evaluasi test
ridge_test_mae, ridge_test_rmse, ridge_test_r2 = hitung_metrics(
    y_test,
    ridge_test_pred
)

In [27]:
print("HASIL RIDGE REGRESSION")

print("\nTrain")
print(f"MAE  : {ridge_train_mae:.4f}")
print(f"RMSE : {ridge_train_rmse:.4f}")
print(f"R2   : {ridge_train_r2:.4f}")

print("\nValidation")
print(f"MAE  : {ridge_val_mae:.4f}")
print(f"RMSE : {ridge_val_rmse:.4f}")
print(f"R2   : {ridge_val_r2:.4f}")

print("\nTest")
print(f"MAE  : {ridge_test_mae:.4f}")
print(f"RMSE : {ridge_test_rmse:.4f}")
print(f"R2   : {ridge_test_r2:.4f}")

HASIL RIDGE REGRESSION

Train
MAE  : 4.6529
RMSE : 5.9776
R2   : 0.6899

Validation
MAE  : 4.3757
RMSE : 5.7335
R2   : 0.6895

Test
MAE  : 4.8698
RMSE : 6.2632
R2   : 0.6481


In [28]:
# Simpan Model Ridge Regression
joblib.dump(
    ridge_model,
    os.path.join(BASE_PATH, 'models', 'ridge_regression.joblib')
)
print("Model berhasil disimpan.")

Model berhasil disimpan.


In [29]:
# Simpan metrics
metrics_ridge = pd.DataFrame({
    "model": ["Ridge Regression"],
    "train_mae": [ridge_train_mae],
    "train_rmse": [ridge_train_rmse],
    "train_r2": [ridge_train_r2],
    "val_mae": [ridge_val_mae],
    "val_rmse": [ridge_val_rmse],
    "val_r2": [ridge_val_r2],
    "test_mae": [ridge_test_mae],
    "test_rmse": [ridge_test_rmse],
    "test_r2": [ridge_test_r2]
})

os.makedirs("results", exist_ok=True)

metrics_ridge.to_csv(
    os.path.join(BASE_PATH, "results", "ridge_regression_metrics.csv"),
    index=False
)
print("Metrics berhasil disimpan.")

Metrics berhasil disimpan.


In [30]:
# Simpan hasil prediksi test
hasil_pred_lr = ID_test.copy()

hasil_pred_lr["actual"] = y_test.values
hasil_pred_lr["prediction_ridge"] = ridge_test_pred

hasil_pred_lr.to_csv(
    os.path.join(BASE_PATH, "results", "ridge_regression_predictions.csv"),
    index=False
)

print("Hasil prediksi berhasil disimpan.")

# preview hasil
print("\nPreview prediction:")
print(hasil_pred_lr.head())

Hasil prediksi berhasil disimpan.

Preview prediction:
         kabupaten_kota   nama_wilayah_bersih  tahun  actual  prediction_ridge
0       Kab. Aceh Barat       kab. aceh barat   2024   54.82         49.432339
1       Kab. Aceh Barat       kab. aceh barat   2025   53.53         50.372778
2  Kab. Aceh Barat Daya  kab. aceh barat daya   2024   55.15         51.402524
3  Kab. Aceh Barat Daya  kab. aceh barat daya   2025   66.77         52.216961
4       Kab. Aceh Besar       kab. aceh besar   2024   51.82         49.803388
